# TensorFlow：GradientTape 与 Keras训练


由 [Vision Learning Hub](https://charlie-sys.github.io/vision-learning-hub/) 提供。
在 Colab 登录自己的Google账号，按从上到下的顺序运行；修改前可“在云端硬盘中保存副本”。
若需要GPU，在“运行时 → 更改运行时类型”中选择可用GPU。免费GPU不保证随时可用，断线后临时文件可能丢失，请下载结果。
本Notebook的结果属于你自己的运行，不应把教程实测数值直接当作本次成绩。


## 1. 环境
TensorFlow是本Notebook的Python框架，不是浏览器TensorFlow.js。GPU不可用时CPU也可完成。


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
print('tensorflow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
tf.keras.utils.set_random_seed(7)


## 2. 同一个梯度例题
应与PyTorch和手算一致：loss=2，dw=4。


In [ ]:
w = tf.Variable(3.)
with tf.GradientTape() as tape:
    loss = (w*2+1-5)**2/2
grad = tape.gradient(loss, w)
print('loss:', float(loss.numpy()), 'gradient:', float(grad.numpy()))
np.testing.assert_allclose(grad.numpy(), 4.)


## 3. 合成二分类数据
两团二维高斯样本，标签不需要下载。训练与验证独立生成；这不是视觉性能基准。


In [ ]:
rng = np.random.default_rng(7)
def data(n):
    y = rng.integers(0, 2, n, dtype=np.int32)
    x = rng.normal(0, .5, (n, 2)).astype('float32') + (2*y[:, None]-1).astype('float32')
    return x, y
x_train, y_train = data(256)
x_val, y_val = data(128)
print(x_train.shape, y_train.shape)


## 4. Keras模型与显式训练循环
输出logits，交叉熵设置from_logits=True；只更新模型的trainable_variables。


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(2),
])
optimizer = tf.keras.optimizers.Adam(.03)
objective = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
history = []
for step in range(60):
    with tf.GradientTape() as tape:
        logits = model(x_train, training=True)
        loss = objective(y_train, logits)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    history.append(float(loss.numpy()))
logits = model(x_val, training=False)
predictions = tf.argmax(logits, axis=-1).numpy()
accuracy = np.mean(predictions == y_val)
print('validation accuracy:', accuracy)
assert accuracy > .9
plt.plot(history)
plt.xlabel('optimization step')
plt.ylabel('cross entropy')
plt.show()


## 5. 保存与恢复


In [ ]:
model.save('tensorflow_classifier.keras')
restored = tf.keras.models.load_model('tensorflow_classifier.keras')
np.testing.assert_allclose(model(x_val, training=False).numpy(), restored(x_val, training=False).numpy(), rtol=1e-5, atol=1e-6)
print('Keras model reload passed')


## 练习
改变数据噪声0.5→1.5，观察错误样本；对照PyTorch里梯度清零与GradientTape的使用差异。
参考：[TensorFlow官方快速入门](https://www.tensorflow.org/tutorials/quickstart/beginner)。
